In [2]:
# baseline model
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import re
import nltk
import string
import numpy as np
import os 
import dagshub
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
df=pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])

In [4]:
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [5]:
# define the preprocessing
nltk.download('wordnet')
nltk.download('stopwords')
def lematization(text):
    lemmatizer=WordNetLemmatizer()
    text=text.split()
    text=[lemmatizer.lemmatize(y) for y in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words=set(stopwords.words('english'))
    Text=[i for i in str(text).split() if i not in stop_words]
    return " ".join(Text)

def removing_numbers(text):
    text=''.join([i for i in text if not i.isdigit()])
    return text

def lower_case(text):
    text=text.split()
    text=[y.lower() for y in text]
    return " ".join(text)
    
def removing_punctuations(text):
    punctuations =  r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""
    # raw string avoids invalid escape warnings
    text= re.sub('[%s]' % re.escape(punctuations), '', text)

    # remove extra whitespace
    text=re.sub(r'\s+', ' ', text)
    text=' '.join(text.split())
    return text.strip()

def removing_urls(text):
    url_pattern=re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_small_sentence(df):
    for i in range(len(df)):
        if len(df.text.iloc[i].split())<3:
            df.text.iloc[i]=np.nan

def normalize_text(df):
    df.content=df.content.apply(lower_case)
    df.content=df.content.apply(remove_stop_words)
    df.content=df.content.apply( removing_numbers)
    df.content=df.content.apply( removing_punctuations)
    df.content=df.content.apply(removing_urls)
    df.content=df.content.apply(lematization)
    return df


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
df=normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [7]:
df.sentiment.value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [8]:
x=df.sentiment.isin(['happiness', 'sadness'])
df=df[x]

In [9]:
df['sentiment']=df['sentiment'].replace({'happiness':1, 'sadness':0})
df.head()

C:\Users\gupta\AppData\Local\Temp\ipykernel_6660\4003209269.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'happiness':1, 'sadness':0})


,sentiment,content
1,0,layin n bed headache ughhhhwaitin call
2,0,funeral ceremonygloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [21]:
# apply the Countvectorizer
vectorizers={
    'Bow': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}
algorithms={
    'Logistic Regression': LogisticRegression(),
    'MultnomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
    }


In [22]:
#
import dagshub

dagshub.init(repo_owner='guptatannu538', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/guptatannu538/mlops-mini-project.mlflow')


Initialized MLflow to track repo "guptatannu538/mlops-mini-project"

Repository guptatannu538/mlops-mini-project initialized!

In [23]:
mlflow.set_experiment('Bow vs Tfidf v3')
with mlflow.start_run(run_name='All Experiment') as parent_run:
    # loop through algorithms and feature extraction methods (Child Runs)
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f'{algo_name} with {vec_name}', nested=True)as child_run:
                X=vectorizer.fit_transform(df['content'])
                y=df['sentiment']
                # split data into train, test
                X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)

                # Log preprocessing parameters
                mlflow.log_param('vectorize', vec_name)
                mlflow.log_param('algorthim', algo_name)
                mlflow.log_param('test_size', 0.2)

                # Model building and training
                model=algorithm
                model.fit(X_train, y_train)
                
                # Log model parameters
                if algo_name=='Logistic Regression':
                    mlflow.log_param("C", model.C)
                elif algo_name=='MultinomialNB':
                    mlflow.log_param('alpha', model.alpha)
                elif algo_name=='XGBoost':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('learning_rate',model.learning_rate)
                elif algo_name=='RandomForest':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('max_depth', model.max_depth)
                elif algo_name=='GradientBoosting':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('learning_rate', model.learning_rate)
                    mlflow.log_param('max_depth', model.max_depth)

                # Model evaluation
                y_pred=model.predict(X_test)
                accuracy=accuracy_score(y_test, y_pred)
                precision=precision_score(y_test, y_pred)
                recall=recall_score(y_test, y_pred)
                f1=f1_score(y_test, y_pred)

                # Log evaluation metrics
                mlflow.log_metric('accuracy', accuracy)
                mlflow.log_metric('precision', precision)
                mlflow.log_metric('recall', recall)
                mlflow.log_metric('f1', f1)

                # Log model
                if algo_name == "XGBoost":
                    mlflow.xgboost.log_model(model, "model")
                else:
                    mlflow.sklearn.log_model(model, "model")

                # Save and log the notebook
                import os
                notebook_path='exp1_bow_vs_tridif.ipynb'
                os.system(f'jupyter nbconvert --to notebook --execute --inplace {notebook_path}')
                mlflow.log_artifact(notebook_path)

                # print the results for verification
                print(f'Accuracy', accuracy)
                print(f'Precision', precision)
                print(f'recall', recall)
                print(f'f1', f1)



2026/07/21 12:50:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.7956626506024096
Precision 0.7844080846968239
recall 0.8029556650246306
f1 0.7935735150925024
🏃 View run Logistic Regression with Bow at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/a1d8f1dfd5804ebd903077f42be7e477
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1


2026/07/21 12:52:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.7913253012048193
Precision 0.7745283018867924
recall 0.8088669950738916
f1 0.7913253012048193
🏃 View run Logistic Regression with TF-IDF at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/2f3e80bc74fb40548f33347f18bfec2a
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1


2026/07/21 12:53:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

Accuracy 0.7686746987951807
Precision 0.7645895153313551
recall 0.761576354679803
f1 0.7630799605133267
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/799876d498114efb96f07943252e88ee
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1


2026/07/21 13:03:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.7195180722891567
Precision 0.8133140376266281
recall 0.5536945812807882
f1 0.6588511137162955
🏃 View run GradientBoosting with Bow at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/1b94695db33d49feab2544a661e9a4a7
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1


2026/07/21 13:04:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.7161445783132531
Precision 0.8078034682080925
recall 0.5507389162561577
f1 0.6549502050380785
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/22ecbb36746440a19685241a2452a8b0
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1
🏃 View run All Experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1/runs/9401458915ec4daebbe866b2c8acb0b9
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/1
